# Practical Application III: Comparing Classifiers

**Overview**: In this practical application, your goal is to compare the performance of the classifiers we encountered in this section, namely K Nearest Neighbor, Logistic Regression, Decision Trees, and Support Vector Machines.  We will utilize a dataset related to marketing bank products over the telephone.  



### Getting Started

Our dataset comes from the UCI Machine Learning repository [link](https://archive.ics.uci.edu/ml/datasets/bank+marketing).  The data is from a Portugese banking institution and is a collection of the results of multiple marketing campaigns.  We will make use of the article accompanying the dataset [here](CRISP-DM-BANK.pdf) for more information on the data and features.



### Problem 1: Understanding the Data

To gain a better understanding of the data, please read the information provided in the UCI link above, and examine the **Materials and Methods** section of the paper.  How many marketing campaigns does this data represent?

17 campaigns

### Problem 2: Read in the Data

Use pandas to read in the dataset `bank-additional-full.csv` and assign to a meaningful variable name.

In [34]:
import pandas as pd

In [35]:
df = pd.read_csv('data/bank-additional-full.csv', sep = ';')

In [36]:
df.head()

,age,job,marital,education,default,housing,loan,contact,month,day_of_week,...,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y
0,56,housemaid,married,basic.4y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
1,57,services,married,high.school,unknown,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
2,37,services,married,high.school,no,yes,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
3,40,admin.,married,basic.6y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
4,56,services,married,high.school,no,no,yes,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no


### Problem 3: Understanding the Features


Examine the data description below, and determine if any of the features are missing values or need to be coerced to a different data type.


```
Input variables:
# bank client data:
1 - age (numeric)
2 - job : type of job (categorical: 'admin.','blue-collar','entrepreneur','housemaid','management','retired','self-employed','services','student','technician','unemployed','unknown')
3 - marital : marital status (categorical: 'divorced','married','single','unknown'; note: 'divorced' means divorced or widowed)
4 - education (categorical: 'basic.4y','basic.6y','basic.9y','high.school','illiterate','professional.course','university.degree','unknown')
5 - default: has credit in default? (categorical: 'no','yes','unknown')
6 - housing: has housing loan? (categorical: 'no','yes','unknown')
7 - loan: has personal loan? (categorical: 'no','yes','unknown')
# related with the last contact of the current campaign:
8 - contact: contact communication type (categorical: 'cellular','telephone')
9 - month: last contact month of year (categorical: 'jan', 'feb', 'mar', ..., 'nov', 'dec')
10 - day_of_week: last contact day of the week (categorical: 'mon','tue','wed','thu','fri')
11 - duration: last contact duration, in seconds (numeric). Important note: this attribute highly affects the output target (e.g., if duration=0 then y='no'). Yet, the duration is not known before a call is performed. Also, after the end of the call y is obviously known. Thus, this input should only be included for benchmark purposes and should be discarded if the intention is to have a realistic predictive model.
# other attributes:
12 - campaign: number of contacts performed during this campaign and for this client (numeric, includes last contact)
13 - pdays: number of days that passed by after the client was last contacted from a previous campaign (numeric; 999 means client was not previously contacted)
14 - previous: number of contacts performed before this campaign and for this client (numeric)
15 - poutcome: outcome of the previous marketing campaign (categorical: 'failure','nonexistent','success')
# social and economic context attributes
16 - emp.var.rate: employment variation rate - quarterly indicator (numeric)
17 - cons.price.idx: consumer price index - monthly indicator (numeric)
18 - cons.conf.idx: consumer confidence index - monthly indicator (numeric)
19 - euribor3m: euribor 3 month rate - daily indicator (numeric)
20 - nr.employed: number of employees - quarterly indicator (numeric)

Output variable (desired target):
21 - y - has the client subscribed a term deposit? (binary: 'yes','no')
```



In [37]:
# Examine data types and missing values
df.info()

# Check for standard missing values (NaN)
print("Missing values (NaN):")
print(df.isnull().sum())

# Check for 'unknown' values in categorical columns
categorical_cols = df.select_dtypes(include='object').columns

print("\nUnknown values:")
for col in categorical_cols:
    count = (df[col] == 'unknown').sum()
    if count > 0:
        print(f"{col}: {count}")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 41188 entries, 0 to 41187
Data columns (total 21 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   age             41188 non-null  int64  
 1   job             41188 non-null  object 
 2   marital         41188 non-null  object 
 3   education       41188 non-null  object 
 4   default         41188 non-null  object 
 5   housing         41188 non-null  object 
 6   loan            41188 non-null  object 
 7   contact         41188 non-null  object 
 8   month           41188 non-null  object 
 9   day_of_week     41188 non-null  object 
 10  duration        41188 non-null  int64  
 11  campaign        41188 non-null  int64  
 12  pdays           41188 non-null  int64  
 13  previous        41188 non-null  int64  
 14  poutcome        41188 non-null  object 
 15  emp.var.rate    41188 non-null  float64
 16  cons.price.idx  41188 non-null  float64
 17  cons.conf.idx   41188 non-null 

### Problem 4: Understanding the Task

After examining the description and data, your goal now is to clearly state the *Business Objective* of the task.  State the objective below.

The business objective is to predict whether a bank client will subscribe to a term deposit as a result of a telephone marketing campaign. By comparing classification models such as K-Nearest Neighbors, Logistic Regression, Decision Trees, and Support Vector Machines, the bank can identify which model best predicts customers who are likely to subscribe. This can help the bank target potential customers more effectively, reduce unnecessary marketing calls, and improve the efficiency and success rate of future marketing campaigns.

In [38]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 41188 entries, 0 to 41187
Data columns (total 21 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   age             41188 non-null  int64  
 1   job             41188 non-null  object 
 2   marital         41188 non-null  object 
 3   education       41188 non-null  object 
 4   default         41188 non-null  object 
 5   housing         41188 non-null  object 
 6   loan            41188 non-null  object 
 7   contact         41188 non-null  object 
 8   month           41188 non-null  object 
 9   day_of_week     41188 non-null  object 
 10  duration        41188 non-null  int64  
 11  campaign        41188 non-null  int64  
 12  pdays           41188 non-null  int64  
 13  previous        41188 non-null  int64  
 14  poutcome        41188 non-null  object 
 15  emp.var.rate    41188 non-null  float64
 16  cons.price.idx  41188 non-null  float64
 17  cons.conf.idx   41188 non-null 

### Problem 5: Engineering Features

Now that you understand your business objective, we will build a basic model to get started.  Before we can do this, we must work to encode the data.  Using just the bank information features, prepare the features and target column for modeling with appropriate encoding and transformations.

In [39]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

# Bank client information features
bank_features = [
    'age',
    'job',
    'marital',
    'education',
    'default',
    'housing',
    'loan'
]

# Features and target
X = df[bank_features]
y = df['y'].map({'no': 0, 'yes': 1})

# Identify categorical and numeric features
categorical_features = [
    'job',
    'marital',
    'education',
    'default',
    'housing',
    'loan'
]

numeric_features = ['age']

# Create preprocessing transformer
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features),
        ('num', 'passthrough', numeric_features)
    ]
)

# Encode/transform the features
X_encoded = preprocessor.fit_transform(X)

print("Original X shape:", X.shape)
print("Encoded X shape:", X_encoded.shape)
print("Target values:")
print(y.value_counts())

Original X shape: (41188, 7)
Encoded X shape: (41188, 34)
Target values:
y
0    36548
1     4640
Name: count, dtype: int64


The six categorical variables need to be converted into numeric form before they can be used by classifiers such as Logistic Regression, KNN, and SVM. OneHotEncoder is appropriate because these categories generally do not have a meaningful numeric order. age is already numeric, so it can remain numeric. 
The target y is converted from: no  -> 0; yes -> 1
The dataset defines y as whether the client subscribed to the term deposit.

### Problem 6: Train/Test Split

With your data prepared, split it into a train and test set.

In [40]:
from sklearn.model_selection import train_test_split

# Split the data into 80% training and 20% testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

X_train shape: (32950, 7)
X_test shape: (8238, 7)
y_train shape: (32950,)
y_test shape: (8238,)


### Problem 7: A Baseline Model

Before we build our first model, we want to establish a baseline.  What is the baseline performance that our classifier should aim to beat?

In [41]:
# Calculate baseline accuracy using the majority class
baseline_accuracy = y_train.value_counts(normalize=True).max()

print(f"Baseline Accuracy: {baseline_accuracy:.4f}")

Baseline Accuracy: 0.8873


### Problem 8: A Simple Model

Use Logistic Regression to build a basic model on your data.  

In [42]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# Define categorical and numeric features
categorical_features = [
    'job',
    'marital',
    'education',
    'default',
    'housing',
    'loan'
]

numeric_features = ['age']

# Preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features),
        ('num', StandardScaler(), numeric_features)
    ]
)

# Create Logistic Regression pipeline
logistic_model = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000))
])

# Train the model
logistic_model.fit(X_train, y_train)

# Make predictions
y_pred = logistic_model.predict(X_test)

# Evaluate accuracy
logistic_accuracy = accuracy_score(y_test, y_pred)

print(f"Logistic Regression Accuracy: {logistic_accuracy:.4f}")

Logistic Regression Accuracy: 0.8874


### Problem 9: Score the Model

What is the accuracy of your model?

In [43]:
# Examine data types and missing values
df.info()

# Check for standard missing values (NaN)
print("Missing values (NaN):")
print(df.isnull().sum())

# Check for 'unknown' values in categorical columns
categorical_cols = df.select_dtypes(include='object').columns

print("\nUnknown values:")
for col in categorical_cols:
    count = (df[col] == 'unknown').sum()
    if count > 0:
        print(f"{col}: {count}")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 41188 entries, 0 to 41187
Data columns (total 21 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   age             41188 non-null  int64  
 1   job             41188 non-null  object 
 2   marital         41188 non-null  object 
 3   education       41188 non-null  object 
 4   default         41188 non-null  object 
 5   housing         41188 non-null  object 
 6   loan            41188 non-null  object 
 7   contact         41188 non-null  object 
 8   month           41188 non-null  object 
 9   day_of_week     41188 non-null  object 
 10  duration        41188 non-null  int64  
 11  campaign        41188 non-null  int64  
 12  pdays           41188 non-null  int64  
 13  previous        41188 non-null  int64  
 14  poutcome        41188 non-null  object 
 15  emp.var.rate    41188 non-null  float64
 16  cons.price.idx  41188 non-null  float64
 17  cons.conf.idx   41188 non-null 

Logistic Regression Result: The Logistic Regression model achieved an accuracy of 88.74%, compared with the baseline accuracy of 88.73%. This indicates that the basic Logistic Regression model provides almost no improvement over simply predicting the majority class (no) for every customer. The result suggests that the seven bank-client demographic and financial features alone may not provide enough information to accurately distinguish customers who will subscribe to a term deposit. Because the target is imbalanced, accuracy alone may also hide poor performance in identifying customers who actually subscribe.

### Problem 10: Model Comparisons

Now, we aim to compare the performance of the Logistic Regression model to our KNN algorithm, Decision Tree, and SVM models.  Using the default settings for each of the models, fit and score each.  Also, be sure to compare the fit time of each of the models.  Present your findings in a `DataFrame` similar to that below:

| Model | Train Time | Train Accuracy | Test Accuracy |
| ----- | ---------- | -------------  | -----------   |
|     |    |.     |.     |

In [44]:
import time
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC

# Define categorical and numeric features
categorical_features = [
    'job',
    'marital',
    'education',
    'default',
    'housing',
    'loan'
]

numeric_features = ['age']

# Preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features),
        ('num', StandardScaler(), numeric_features)
    ]
)

# Models using default settings
models = {
    'Logistic Regression': LogisticRegression(),
    'KNN': KNeighborsClassifier(),
    'Decision Tree': DecisionTreeClassifier(),
    'SVM': SVC()
}

results = []

for name, model in models.items():

    # Build pipeline
    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('classifier', model)
    ])

    # Measure training time
    start_time = time.time()
    pipeline.fit(X_train, y_train)
    train_time = time.time() - start_time

    # Calculate accuracy
    train_accuracy = pipeline.score(X_train, y_train)
    test_accuracy = pipeline.score(X_test, y_test)

    results.append({
        'Model': name,
        'Train Time': train_time,
        'Train Accuracy': train_accuracy,
        'Test Accuracy': test_accuracy
    })

# Create comparison DataFrame
results_df = pd.DataFrame(results)

# Round values for readability
results_df = results_df.round({
    'Train Time': 4,
    'Train Accuracy': 4,
    'Test Accuracy': 4
})

results_df

,Model,Train Time,Train Accuracy,Test Accuracy
0,Logistic Regression,0.0493,0.8873,0.8874
1,KNN,0.0218,0.8903,0.8785
2,Decision Tree,0.1878,0.9171,0.8651
3,SVM,29.1888,0.8882,0.8864


Model Comparison: Logistic Regression produced the highest test accuracy at 88.74%, slightly exceeding the baseline accuracy of 88.73%. SVM achieved a similar test accuracy of 88.64%, but required approximately 30 seconds to train, making it substantially more computationally expensive. KNN achieved 87.85% test accuracy, while the Decision Tree had the lowest test accuracy at 86.44%.
The Decision Tree also showed the largest difference between training accuracy (91.71%) and test accuracy (86.44%), suggesting that it is overfitting the training data. Logistic Regression provides the best balance of test accuracy and computational efficiency among the four default models. However, none of the models meaningfully outperform the 88.73% baseline, suggesting that the seven bank-client features alone have limited predictive power.

### Problem 11: Improving the Model

Now that we have some basic models on the board, we want to try to improve these.  Below, we list a few things to explore in this pursuit.


- Hyperparameter tuning and grid search.  All of our models have additional hyperparameters to tune and explore.  For example the number of neighbors in KNN or the maximum depth of a Decision Tree.  
- Adjust your performance metric

In [45]:
import time
import pandas as pd

from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Models and parameter grids
# Smaller grids to avoid excessive CPU/memory usage

models_params = {
    'Logistic Regression': {
        'model': LogisticRegression(max_iter=1000),
        'params': {
            'classifier__C': [0.1, 1],
            'classifier__class_weight': [None, 'balanced']
        }
    },

    'KNN': {
        'model': KNeighborsClassifier(),
        'params': {
            'classifier__n_neighbors': [3, 5],
            'classifier__weights': ['uniform', 'distance']
        }
    },

    'Decision Tree': {
        'model': DecisionTreeClassifier(random_state=42),
        'params': {
            'classifier__max_depth': [3, 5],
            'classifier__class_weight': [None, 'balanced']
        }
    },

    'SVM': {
        'model': SVC(),
        'params': {
            'classifier__C': [0.1, 1],
            'classifier__class_weight': [None, 'balanced']
        }
    }
}

In [46]:
results = []

for name, config in models_params.items():

    print(f"\nTuning {name}...")

    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('classifier', config['model'])
    ])

    grid = GridSearchCV(
        pipeline,
        config['params'],
        cv=2,
        scoring='f1',
        n_jobs=1
    )

    start = time.time()

    grid.fit(X_train, y_train)

    elapsed = time.time() - start

    y_pred = grid.predict(X_test)

    results.append({
        'Model': name,
        'Best Parameters': grid.best_params_,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Recall': recall_score(y_test, y_pred),
        'F1 Score': f1_score(y_test, y_pred),
        'Tuning Time': elapsed
    })

    print("Best parameters:", grid.best_params_)
    print(f"Finished in {elapsed:.2f} seconds")


Tuning Logistic Regression...
Best parameters: {'classifier__C': 0.1, 'classifier__class_weight': 'balanced'}
Finished in 0.54 seconds

Tuning KNN...
Best parameters: {'classifier__n_neighbors': 3, 'classifier__weights': 'distance'}
Finished in 43.47 seconds

Tuning Decision Tree...
Best parameters: {'classifier__class_weight': 'balanced', 'classifier__max_depth': 5}
Finished in 0.28 seconds

Tuning SVM...
Best parameters: {'classifier__C': 0.1, 'classifier__class_weight': 'balanced'}
Finished in 103.27 seconds


In [47]:
tuned_results = pd.DataFrame(results)

tuned_results[
    ['Model', 'Accuracy', 'Precision',
     'Recall', 'F1 Score', 'Tuning Time']
].round(4)

,Model,Accuracy,Precision,Recall,F1 Score,Tuning Time
0,Logistic Regression,0.5829,0.1581,0.6250,0.2524,0.5426
1,KNN,0.8602,0.2282,0.1013,0.1403,43.4672
2,Decision Tree,0.6966,0.1806,0.4784,0.2622,0.2830
3,SVM,0.6028,0.1642,0.6175,0.2594,103.2748


Because the target variable is imbalanced, I used F1 score as the primary metric for hyperparameter tuning rather than accuracy alone. GridSearchCV with cross-validation was used to tune Logistic Regression, KNN, Decision Tree, and SVM.

After tuning, the Decision Tree achieved the highest F1 score of 0.2622, with a recall of 0.4784 and precision of 0.1806. SVM produced a similar F1 score of 0.2594 and higher recall of 0.6175, but required substantially more computation time. Logistic Regression achieved the highest recall of 0.6250 but had lower precision and an F1 score of 0.2524.

KNN had the highest accuracy at 0.8602, but its recall was only 0.1013. This demonstrates why accuracy alone is not an appropriate metric for this imbalanced classification problem: a model can achieve relatively high accuracy while failing to identify most customers who actually subscribe.

Based on F1 score and computational efficiency, the tuned Decision Tree provides the best overall balance among the models tested. If the business places a higher priority on identifying as many potential subscribers as possible, Logistic Regression or SVM could instead be considered because of their higher recall.

##### Questions